In [17]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.layers import GlobalAveragePooling1D
df = pd.read_csv("processed_music_dataset2.csv")

kmeans = joblib.load("kmeans_model.pkl")
scaler = joblib.load("scaler.pkl")


In [18]:
print(df.head())
print(df.shape)

                  artists                  track_name  popularity  \
0             Gen Hoshino                      Comedy          73   
1            Ben Woodward            Ghost - Acoustic          55   
2  Ingrid Michaelson;ZAYN              To Begin Again          57   
3            Kina Grannis  Can't Help Falling In Love          71   
4        Chord Overstreet                     Hold On          82   

   duration_ms  explicit  danceability  energy  key  loudness  mode  ...  \
0       230666         0         0.676  0.4610    1    -6.746     0  ...   
1       149610         0         0.420  0.1660    1   -17.235     1  ...   
2       210826         0         0.438  0.3590    0    -9.734     1  ...   
3       201933         0         0.266  0.0596    0   -18.515     1  ...   
4       198853         0         0.618  0.4430    2    -9.681     1  ...   

   acousticness  instrumentalness  liveness  valence    tempo  time_signature  \
0        0.0322          0.000001    0.3580    

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81344 entries, 0 to 81343
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   artists           81344 non-null  object 
 1   track_name        81344 non-null  object 
 2   popularity        81344 non-null  int64  
 3   duration_ms       81344 non-null  int64  
 4   explicit          81344 non-null  int64  
 5   danceability      81344 non-null  float64
 6   energy            81344 non-null  float64
 7   key               81344 non-null  int64  
 8   loudness          81344 non-null  float64
 9   mode              81344 non-null  int64  
 10  speechiness       81344 non-null  float64
 11  acousticness      81344 non-null  float64
 12  instrumentalness  81344 non-null  float64
 13  liveness          81344 non-null  float64
 14  valence           81344 non-null  float64
 15  tempo             81344 non-null  float64
 16  time_signature    81344 non-null  int64 

In [20]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['song_id'] = encoder.fit_transform(df['track_name'])

In [21]:
song_mapping = dict(zip(df.song_id, df.track_name))

In [22]:
import random

song_ids = df['song_id'].tolist()

user_sequences = []

for i in range(1000):

    seq = random.sample(song_ids, 20)

    user_sequences.append(seq)

In [23]:
sequence_length = 5

X = []
y = []

for seq in user_sequences:

    for i in range(len(seq) - sequence_length):

        X.append(seq[i:i+sequence_length])
        y.append(seq[i+sequence_length])

X = np.array(X)
# y = np.array(y)
y = df.iloc[y]['cluster'].values

In [24]:
# Build Transformer Model

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, MultiHeadAttention
from tensorflow.keras.layers import Dense, LayerNormalization
from tensorflow.keras.models import Model

In [25]:
sequence_length = 5
num_songs = df['song_id'].nunique()
num_classes = 10 

inputs = Input(shape=(sequence_length,))

embedding = Embedding(num_songs, 64)(inputs)

attention = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(embedding, embedding)

attention = LayerNormalization()(attention)

dense = Dense(128, activation="relu")(attention)

dense = Dense(64, activation="relu")(dense)
pooled = GlobalAveragePooling1D()(dense)

output = Dense(num_classes, activation="softmax")(pooled)

model = Model(inputs, output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [26]:
model.fit(
    X,
    y,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 14s 67ms/step - accuracy: 0.2131 - loss: 2.1323 - val_accuracy: 0.2110 - val_loss: 2.1283
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - accuracy: 0.2584 - loss: 2.0023 - val_accuracy: 0.1947 - val_loss: 2.1516
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.3693 - loss: 1.6313 - val_accuracy: 0.1220 - val_loss: 2.3539
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 58ms/step - accuracy: 0.4868 - loss: 1.2463 - val_accuracy: 0.1293 - val_loss: 2.7059
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 58ms/step - accuracy: 0.6278 - loss: 0.8995 - val_accuracy: 0.1203 - val_loss: 3.3938
Epoch 6/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.7502 - loss: 0.6135 - val_accuracy: 0.1447 - val_loss: 4.3240
Epoch 7/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 12s 62ms/step - accuracy: 0.8375 - loss: 0.4092 - val_accuracy: 0.1317 - val_loss: 5.2859
Epoch 8/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 60ms/step - accuracy: 0.8897 - loss: 0.2893 - 

In [27]:
model.save("transformer_model.h5")

In [28]:
test_seq = np.array([X[0]])

prediction = model.predict(test_seq)

next_song_id = np.argmax(prediction)

print(song_mapping[next_song_id])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
"DEVILS NEVER CRY"(スタッフロール)


In [29]:
def hybrid_recommend(song_index, user_mood):

    input_song = df.iloc[song_index]

    input_genre = input_song['track_genre']

    song_vector = X_scaled[song_index].reshape(1,-1)

    transformer_cluster = predict_cluster(song_index)

    cluster_df = df[df['cluster'] == transformer_cluster]

    # genre filtering
    cluster_df = cluster_df[cluster_df['track_genre'] == input_genre]

    cluster_vectors = X_scaled[cluster_df.index]

    similarity = cosine_similarity(song_vector, cluster_vectors)[0]

    results = []

    for i, sim in enumerate(similarity):

        song = cluster_df.iloc[i]

        fuzzy = 1 if song['mood'] == user_mood else 0

        final_score = 0.5*sim + 0.3*1 + 0.2*fuzzy

        results.append((song['track_name'], song['artists'], final_score))

    results.sort(key=lambda x: x[2], reverse=True)

    return results[:10]